# Clase 10 — Embeddings y similitud semántica
### De los documentos a un buscador que entiende el significado

**Procesamiento del Lenguaje Natural — Nivel terciario**  
**Clase completa en este Colab:** no hace falta PowerPoint. Primero leemos las explicaciones y después ejecutamos las celdas.

**La idea en una oración:** *vamos a enseñarle al sistema a encontrar qué parte de nuestros documentos está relacionada con una pregunta, incluso si usa otras palabras.*

> **Importante:** hoy **NO generamos la respuesta final con el chatbot**. Hoy construimos la **búsqueda de información** que va antes de esa respuesta. En la clase 11 uniremos este buscador con el modelo generativo.


## Qué deberíamos entender al terminar

1. Explicar con nuestras palabras qué es un **embedding**.
2. Distinguir el **modelo de embeddings** del **modelo de IA que redacta respuestas**.
3. Cargar el `fragmentos_proyecto.json` de la clase 9.
4. Convertir los fragmentos y una pregunta en vectores de números.
5. Compararlos usando **similitud coseno**.
6. Elegir qué fragmentos recuperar mediante `top_k` y `umbral`.
7. Identificar **qué código usaremos después en el chatbot final**.

**Plan de clase (aprox. 80 minutos):** 20 min de explicación + 35 min de práctica + 15 min de pruebas + 10 min de evaluación y cierre.


# PARTE A — TEORÍA (leer antes de ejecutar código)

## 1. ¿Qué venimos construyendo desde la clase 7?

| Clase | Lo que hicimos | Archivo o resultado |
|---|---|---|
| 7 | Elegimos y evaluamos fuentes de conocimiento | PDFs, Word, etc. |
| 8 | Leímos los archivos y limpiamos el texto | `documentos_proyecto.json` |
| 9 | Dividimos ese texto en partes más pequeñas | `fragmentos_proyecto.json` |
| **10 (hoy)** | Convertimos cada fragmento en números y buscamos cuáles se parecen a una pregunta | **embeddings + buscador semántico** |
| 11 (después) | Entregamos los fragmentos encontrados a la IA para que redacte una respuesta | **RAG completo** |

**¿Por qué no buscamos directamente en el PDF?** Porque ya convertimos el contenido a texto legible y lo dividimos. Ahora podemos buscar en partes concretas, en vez de enviar un documento entero.

**¿Por qué usamos JSON?** Porque conserva cada fragmento y sus datos (`fuente`, `texto`, `fragmento_numero`). El CSV es útil para revisar tablas, pero **no reemplaza al JSON** que recibe esta clase.


## 2. Problema: buscar palabras exactas no alcanza

Imaginemos que un PDF dice:

> *“Para realizar la inscripción, el aspirante debe presentar su documentación.”*

Pero la persona pregunta:

> *“¿Qué papeles tengo que llevar para anotarme?”*

Una búsqueda básica de palabras podría fallar porque el PDF dice **inscripción/documentación** y el usuario dice **anotarme/papeles**.

**Para una persona, las dos frases hablan de lo mismo.** Queremos que nuestro buscador también lo detecte.

Ahí aparecen los **embeddings**.


## 3. ¿Qué es un embedding? (explicación sencilla)

Un **embedding** es una **representación numérica de un texto**. En lugar de guardar únicamente una frase con letras, un **modelo de embeddings** la convierte en una lista de muchos números llamada **vector**.

La idea es que **textos de significado parecido suelen obtener vectores cercanos**.

**Ejemplo ilustrativo, NO generado por el modelo real:**

| Texto | Vector inventado para entender la idea |
|---|---|
| “Quiero anotarme a la carrera” | `[0.7, 0.2, 0.9]` |
| “¿Cómo hago la inscripción?” | `[0.72, 0.19, 0.88]` |
| “¿Cuándo pasa el colectivo?” | `[0.1, 0.95, 0.14]` |

Los dos primeros vectores son parecidos en este ejemplo porque las frases tienen un sentido parecido. El tercero representa otro tema.

**Atención:** esos tres números son inventados para explicar el concepto. El modelo real crea vectores mucho más largos, y **cada número individual no equivale a una palabra ni tiene una interpretación simple**. Es el vector completo el que permite comparar significados.

**Frase para recordar:** *embedding = transformar un texto en números para poder compararlo con otros textos por su significado*.


## 4. Tenemos DOS modelos con tareas diferentes

| Modelo | Qué hace | ¿Responde la pregunta final? |
|---|---|---|
| **Modelo de embeddings**: `intfloat/multilingual-e5-small` | Transforma la pregunta y los fragmentos en vectores; ayuda a encontrar información relacionada. | **No** |
| **Modelo generativo** (por ejemplo, Qwen en nuestras clases anteriores) | Lee la pregunta y el contexto recuperado; **redacta** la respuesta. | **Sí, cuando lo integremos** |

**Ejemplo:**

- Usuario pregunta: “¿Qué necesito para anotarme?”.
- **Modelo de embeddings:** encuentra el fragmento del PDF que explica la inscripción.
- **Modelo generativo:** con ese fragmento, responde de forma clara en español.

> El modelo de embeddings **no busca en Internet**, **no redacta una respuesta** y **no sabe si un reglamento es oficial**. Solo ayuda a comparar textos. La calidad y confiabilidad de la fuente debe revisarse aparte.


## 5. Así viaja una pregunta por nuestro futuro chatbot

```text
FUENTES (PDF, DOCX, TXT)
        │  Clase 8: extraer y limpiar texto
        ▼
documentos_proyecto.json
        │  Clase 9: fragmentar
        ▼
fragmentos_proyecto.json
        │  Clase 10: convertir cada fragmento en un embedding
        ▼
VECTORES DE LOS FRAGMENTOS

PREGUNTA DEL USUARIO
        │  Clase 10: convertir la pregunta en un embedding
        ▼
VECTOR DE LA PREGUNTA
        │  comparar con los vectores de los fragmentos
        ▼
FRAGMENTOS MÁS RELACIONADOS CON LA PREGUNTA
        │  Clase 11: armar contexto y enviarlo al modelo generativo
        ▼
RESPUESTA CON IA Y REFERENCIA A LAS FUENTES
```

**Nombre del proceso completo:** RAG (*generación aumentada por recuperación*).

La clase 10 construye solamente la parte **recuperación** del RAG. Por eso **si hoy vemos fragmentos y porcentajes o puntajes de similitud, NO significa que el chatbot ya esté contestando**.


## 6. ¿Qué es la similitud coseno?

Tenemos un vector para la pregunta y un vector para cada fragmento. Necesitamos una forma de medir **qué tan orientados en la misma dirección están**. Para eso usamos la **similitud coseno**.

- Un resultado **más cercano a 1** generalmente indica mayor parecido entre los vectores.
- Un resultado **más bajo** indica menor parecido.
- Matemáticamente puede tomar valores entre **-1 y 1**; en búsquedas con este modelo veremos habitualmente otros rangos según los textos.

**Muy importante:** una similitud de `0.75` **NO quiere decir “75 % de respuesta correcta”**. No confirma que el dato sea verdadero ni que el documento sea confiable. Solo indica qué tan relacionados encontró los textos este modelo.

**Ejemplo ficticio de resultados:**

| Pregunta: “¿Cómo me inscribo?” | Similitud inventada |
|---|---:|
| Fragmento del proceso de inscripción | 0.83 |
| Fragmento de horarios de secretaría | 0.49 |
| Fragmento de préstamo de libros | 0.18 |

Aquí **revisaríamos primero** el fragmento de inscripción. Los números reales los calculará el código más abajo.


## 7. Dos parámetros muy importantes: `top_k` y `umbral`

**`top_k`: cuántos resultados como máximo queremos recuperar.**  
Si `top_k=3`, el sistema considera los **3 fragmentos más relacionados** con la pregunta. Si existen menos de 3, recupera los disponibles.

**`umbral`: similitud mínima para aceptar un resultado.**  
Si `umbral=0.75`, descartamos los resultados con puntaje menor que 0.75.

| Configuración | Qué puede pasar |
|---|---|
| `top_k=1` | Trae poca información; puede quedarse corto. |
| `top_k=4` | Trae más contexto, pero puede incorporar información irrelevante. |
| Umbral bajo | Aparecen más fragmentos, incluso de poca relación. |
| Umbral alto | Filtra más, pero puede dejar preguntas sin ningún resultado. |

### Advertencia MUY importante sobre el modelo que usamos

**No todos los modelos producen puntajes parecidos.** El modelo `multilingual-e5-small` suele producir similitudes coseno aproximadamente entre **0.7 y 1.0**, incluso para textos que no son idénticos. Por eso, usar un umbral de `0.25` o `0.35` (como en algunos ejemplos genéricos) puede **no filtrar casi nada** con este modelo.

Para esta clase empezamos con **`0.75` solo como valor de prueba** y comparamos con `0.80`, `0.85` y `0.90`. **No existe un umbral universal que garantice respuestas correctas**. Lo ajustamos mirando resultados reales y comprobando si los fragmentos contienen la información necesaria.

**Un puntaje de `0.85` NO quiere decir 85 % de verdad ni 85 % de confianza.** Lo que más importa es qué fragmento quedó por encima de los demás y si efectivamente ayuda a responder.

**Fuente técnica para consultar:** [ficha oficial de multilingual-e5-small en Hugging Face](https://huggingface.co/intfloat/multilingual-e5-small) (preguntas frecuentes: prefijos `query:`/`passage:` y rango habitual de las similitudes).


## 8. ¿Qué parte del código va a usarse en el chatbot final?

**Esta es la conexión más importante de la clase.**

| Código que veremos hoy | ¿Se reutiliza en el chatbot? | Para qué se usa |
|---|---|---|
| Cargar `fragmentos_proyecto.json` | **Sí** | Tener disponibles los textos y sus fuentes. |
| `SentenceTransformer(...)` | **Sí** | Cargar el modelo de embeddings. |
| `embeddings_fragmentos = modelo_embeddings.encode(...)` | **Sí** | Preparar la búsqueda de información. Se calcula al cargar/actualizar la base documental, **no en cada pregunta**. |
| `buscar_fragmentos(pregunta, ...)` | **Sí, es la pieza central de esta clase** | Recuperar los fragmentos relevantes para cada consulta. |
| `construir_contexto(...)` | **Sí** | Preparar el texto que verá la IA generativa. |
| Listas de `preguntas_prueba` / tablas / comparaciones | **No necesariamente** | Son herramientas para practicar y evaluar. |
| Exportación del CSV de evaluación | **No es la respuesta del chatbot** | Es evidencia para revisar si la búsqueda funcionó. |

**En la clase 11 agregaremos lo que falta:** llamar al modelo generativo con `pregunta + fragmentos recuperados` y mostrar la respuesta con fuentes. **Hoy no entrenamos un modelo nuevo; usamos un modelo de embeddings ya entrenado.**


---
# PARTE B — PRÁCTICA PASO A PASO

### Antes de empezar

**Si tenés el archivo `fragmentos_proyecto.json` de la clase 9:** usalo. **No subas** `documentos_proyecto.json`, `fragmentos_proyecto.csv` ni PDFs: esta clase necesita el **JSON de fragmentos**.

**Si todavía no lo tenés:** hay un **modo demostración** para que puedas aprender sin rehacer las clases anteriores. Los datos de demostración son inventados y no sirven como información institucional real.

**Cómo ejecutar en Colab:** Entorno de ejecución → Ejecutar todas, o ejecutar cada celda con el botón ▶ de arriba hacia abajo.


## Paso 1 — Instalar la biblioteca de embeddings

La biblioteca `sentence-transformers` nos permite cargar modelos que convierten texto en vectores. `pandas` nos sirve para mostrar resultados en tablas y `torch` se utiliza en las operaciones numéricas (vienen disponibles en el entorno habitual de Colab).

**Esto es preparación del entorno. No es todavía el buscador.**


In [ ]:
# Ejecutar esta celda una vez por sesión de Colab.
!pip -q install sentence-transformers


## Paso 2 — Importar las herramientas

- `json`: leer los fragmentos de la clase 9.
- `pandas`: tablas para inspeccionar.
- `torch`: trabajar con los vectores.
- `F.cosine_similarity`: comparar los vectores.
- `SentenceTransformer`: modelo que transforma texto en embeddings.


In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from sentence_transformers import SentenceTransformer

print("¿Hay GPU disponible?", torch.cuda.is_available())
print("No es obligatoria: este modelo pequeño también funciona con CPU.")


## Paso 3 — Cargar los fragmentos que vienen de la clase 9

**¿Qué tiene el JSON?** Una lista de objetos. Cada objeto corresponde a una parte del documento:

```json
{
  "fuente": "reglamento.pdf",
  "fragmento_numero": 1,
  "texto": "Contenido del fragmento..."
}
```

Los datos **no se convierten todavía en embeddings**. Primero los cargamos y validamos.

### Elegí una opción

- **Mi proyecto:** dejá `USAR_DEMO = False` y subí el `fragmentos_proyecto.json` de clase 9.
- **Prueba sin archivos:** cambiá a `USAR_DEMO = True`. Usaremos fragmentos ficticios preparados para esta clase.


In [ ]:
USAR_DEMO = False  # Cambiá a True SOLO si todavía no tenés el JSON de la clase 9.

fragmentos_demo = [
    {"fuente": "guia_inscripcion_DEMO.txt", "fragmento_numero": 1,
     "texto": "Para realizar la inscripción, el aspirante debe completar el formulario de ingreso y presentar la documentación solicitada por Secretaría. Los requisitos definitivos deben consultarse en la comunicación oficial."},
    {"fuente": "guia_inscripcion_DEMO.txt", "fragmento_numero": 2,
     "texto": "Las fechas para anotarse se anuncian a través del calendario académico vigente. La institución informa los plazos por sus canales oficiales; este texto de ejemplo no contiene fechas reales."},
    {"fuente": "reglamento_DEMO.txt", "fragmento_numero": 1,
     "texto": "La regularidad de las materias se rige por el reglamento académico vigente. El alumnado debe consultar ese documento para confirmar duración, condiciones de asistencia y evaluaciones."},
    {"fuente": "secretaria_DEMO.txt", "fragmento_numero": 1,
     "texto": "Secretaría atiende consultas sobre trámites administrativos, constancias, documentación, certificados y cuestiones relacionadas con inscripciones. Los horarios deben verificarse en un aviso actualizado."},
    {"fuente": "biblioteca_DEMO.txt", "fragmento_numero": 1,
     "texto": "La biblioteca orienta sobre consulta de libros, préstamo de material, devoluciones y disponibilidad del catálogo. Las condiciones de préstamo deben verificarse con su reglamento."},
    {"fuente": "becas_DEMO.txt", "fragmento_numero": 1,
     "texto": "Las becas estudiantiles pueden requerir una solicitud y documentación respaldatoria. Las convocatorias, requisitos concretos y fechas cambian, por eso deben consultarse en fuentes oficiales."}
]

if USAR_DEMO:
    fragmentos = fragmentos_demo
    print("MODO DEMO: se cargaron fragmentos ficticios, solo para aprender.")
else:
    from google.colab import files
    print("Subí el archivo fragmentos_proyecto.json (NO PDF ni CSV).")
    subidos = files.upload()
    candidatos = [nombre for nombre in subidos if nombre.lower().endswith(".json")]
    if len(candidatos) != 1:
        raise ValueError("Subí exactamente un archivo JSON de fragmentos de la clase 9.")
    with open(candidatos[0], "r", encoding="utf-8") as archivo:
        fragmentos = json.load(archivo)

if not isinstance(fragmentos, list) or len(fragmentos) == 0:
    raise ValueError("El archivo debe contener una lista NO vacía de fragmentos.")

for posicion, frag in enumerate(fragmentos, start=1):
    if not isinstance(frag, dict) or not isinstance(frag.get("texto"), str):
        raise ValueError(f"El fragmento {posicion} no tiene un campo 'texto' válido.")
    if not frag["texto"].strip():
        raise ValueError(f"El fragmento {posicion} tiene texto vacío.")
    if not frag.get("fuente"):
        raise ValueError(f"El fragmento {posicion} no indica la fuente.")
    frag.setdefault("fragmento_numero", posicion)

print("Cantidad de fragmentos:", len(fragmentos))
display(pd.DataFrame(fragmentos)[["fuente", "fragmento_numero", "texto"]].head())


### Atención: ¿qué documentos debemos indexar?

El buscador considera los textos que le pasemos, pero **no puede decidir por sí solo si una fuente es oficial o está desactualizada**. Si en clases anteriores detectaron un documento dudoso o marcado para descartar, **no debería formar parte de la base de respuestas**.

Para practicar, pueden excluir una fuente indicando su nombre exacto abajo. Si no tienen ninguna para excluir, no cambien nada. **Esto no reemplaza la revisión manual de confiabilidad.**


In [ ]:
# Escribir nombres EXACTOS de fuentes que el grupo decidió excluir.
FUENTES_EXCLUIDAS = set()  # Ejemplo: {"mi_fuente_dudosa.pdf"}

fragmentos = [f for f in fragmentos if f["fuente"] not in FUENTES_EXCLUIDAS]
if not fragmentos:
    raise ValueError("No quedaron fragmentos para buscar. Revisá las fuentes excluidas.")

print("Fragmentos que se utilizarán en la búsqueda:", len(fragmentos))
print("Fuentes incluidas:", sorted({f["fuente"] for f in fragmentos}))


## Paso 4 — Cargar el modelo de embeddings (NO el modelo que responde)

Usaremos **`intfloat/multilingual-e5-small`**:

- Es multilingüe: permite trabajar con consultas en español.
- Fue entrenado para representar textos de modo que podamos compararlos.
- Es relativamente liviano para ejecutar en Colab.
- No genera una respuesta conversacional; para eso más adelante usaremos el modelo generativo.

**La primera vez puede descargar archivos del modelo desde Hugging Face.**

La familia E5 utiliza dos prefijos importantes: **`passage:`** para los fragmentos y **`query:`** para la pregunta. **No son las respuestas**: son indicaciones que el modelo de embeddings usa al procesar cada tipo de texto.


In [ ]:
MODELO_EMBEDDINGS = "intfloat/multilingual-e5-small"
modelo_embeddings = SentenceTransformer(MODELO_EMBEDDINGS)
print("Modelo de embeddings cargado:", MODELO_EMBEDDINGS)
print("Dimensiones del vector:", modelo_embeddings.get_sentence_embedding_dimension())


## Paso 5 — Convertir los fragmentos en vectores

Esto se hace **una vez cuando preparamos o actualizamos nuestros documentos**. No tiene sentido transformar todos los documentos otra vez por cada pregunta si la base no cambió.

Vamos a crear dos cosas distintas:

1. `fragmentos`: los **textos originales** y el nombre de sus fuentes.
2. `embeddings_fragmentos`: sus **vectores numéricos**.

**¿Por qué guardamos ambos?** Los números sirven para comparar, pero después **necesitamos el texto real** para dárselo a la IA y mostrar la fuente.


In [ ]:
# La palabra "passage:" le indica al modelo que estos textos son documentos.
textos_fragmentos = ["passage: " + frag["texto"] for frag in fragmentos]

embeddings_fragmentos = modelo_embeddings.encode(
    textos_fragmentos,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Fragmentos:", len(fragmentos))
print("Forma de la matriz:", tuple(embeddings_fragmentos.shape))
print("Primeros 8 números del primer embedding (solo para mirar):")
print(embeddings_fragmentos[0][:8].tolist())


### Entender la salida que acabamos de ver

Si aparece una forma como `(6, 384)` (en el modo demostración), significa:

- **6 filas:** 6 fragmentos de texto.
- **384 columnas:** cada fragmento fue transformado en un vector de 384 números.

**NO significa** que hay 384 palabras, 384 preguntas, ni 384 respuestas. Son dimensiones de la representación numérica.

En tus fuentes reales la primera cantidad cambiará según cuántos fragmentos hayas creado en la clase 9.

**`normalize_embeddings=True`** hace que los vectores estén normalizados y facilita compararlos. Se mantiene tanto para los fragmentos como para las preguntas.


## Paso 6 — Una prueba muy pequeña: mismo significado, palabras diferentes

Antes de buscar en todos los PDFs, vamos a comprobar la idea con **tres frases simples**.

Buscamos qué frase se parece más a la pregunta **“¿Qué papeles debo llevar para anotarme?”**.

No son datos del proyecto: es una **demostración del significado de embedding**.


In [ ]:
pregunta_ejemplo = "¿Qué papeles debo llevar para anotarme?"
frases_ejemplo = [
    "Para la inscripción debés presentar la documentación requerida.",
    "La biblioteca tiene libros disponibles para préstamo.",
    "Los colectivos urbanos circulan por diferentes barrios."
]

vector_pregunta_ejemplo = modelo_embeddings.encode(
    ["query: " + pregunta_ejemplo], convert_to_tensor=True, normalize_embeddings=True
)
vectores_frases_ejemplo = modelo_embeddings.encode(
    ["passage: " + frase for frase in frases_ejemplo],
    convert_to_tensor=True, normalize_embeddings=True
)
puntajes_ejemplo = F.cosine_similarity(vector_pregunta_ejemplo, vectores_frases_ejemplo, dim=1)

df_ejemplo = pd.DataFrame({"frase": frases_ejemplo, "similitud": puntajes_ejemplo.tolist()})
display(df_ejemplo.sort_values("similitud", ascending=False))
print("Observá: aunque cambian las palabras, buscamos similitud de significado.")


## Paso 7 — Crear el BUSCADOR SEMÁNTICO (lo más importante del código)

La función que sigue se llama **`buscar_fragmentos()`**. Es la que **reutilizaremos en el chatbot final**.

Cuando la persona envíe una pregunta, esta función:

1. Recibe **la pregunta original** del usuario.
2. Crea el **embedding de la pregunta** usando el mismo modelo (`query:`).
3. Compara ese vector con los vectores de **TODOS los fragmentos**.
4. Ordena los resultados del más relacionado al menos relacionado.
5. Se queda con un máximo de `top_k` resultados.
6. Descarta los que tienen una similitud inferior al `umbral`.
7. Devuelve una tabla con **texto + fuente + puntuación**.

**No genera una respuesta.** Devuelve la información que necesitará el modelo generativo.


In [ ]:
COLUMNAS_RESULTADOS = ["pregunta", "fuente", "fragmento_numero", "similitud", "texto"]


def buscar_fragmentos(pregunta: str, top_k: int = 4, umbral: float = 0.75) -> pd.DataFrame:
    """Recupera fragmentos relacionados con una pregunta; NO genera una respuesta con IA."""
    if not isinstance(pregunta, str) or not pregunta.strip():
        return pd.DataFrame(columns=COLUMNAS_RESULTADOS)
    if top_k < 1:
        raise ValueError("top_k debe ser al menos 1")

    # 1. Transformamos la PREGUNTA en un vector de números.
    embedding_pregunta = modelo_embeddings.encode(
        ["query: " + pregunta],
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    # 2. Comparamos la pregunta con cada FRAGMENTO (que ya tiene su embedding).
    similitudes = F.cosine_similarity(
        embedding_pregunta, embeddings_fragmentos, dim=1
    )

    # 3. Elegimos como máximo top_k fragmentos con mayor similitud.
    cantidad = min(int(top_k), len(fragmentos))
    valores, indices = torch.topk(similitudes, k=cantidad)

    # 4. Recuperamos el TEXTO ORIGINAL y el nombre de su FUENTE.
    resultados = []
    for valor, indice in zip(valores.tolist(), indices.tolist()):
        if valor >= umbral:
            frag = fragmentos[indice]
            resultados.append({
                "pregunta": pregunta,
                "fuente": frag["fuente"],
                "fragmento_numero": frag["fragmento_numero"],
                "similitud": round(float(valor), 4),
                "texto": frag["texto"]  # completo: lo usaremos después como contexto
            })

    # Conservamos las columnas aun si la pregunta no tiene resultados.
    return pd.DataFrame(resultados, columns=COLUMNAS_RESULTADOS)


### ¿En qué parte de la función está cada tarea?

| Línea del código | Para qué sirve |
|---|---|
| `modelo_embeddings.encode(["query: " + pregunta], ...)` | Convierte **la pregunta** en números. |
| `F.cosine_similarity(...)` | Calcula qué tan parecida es la pregunta a cada fragmento. |
| `torch.topk(...)` | Obtiene los **más relacionados**. |
| `if valor >= umbral:` | Evita aceptar resultados por debajo del mínimo elegido. |
| `frag["texto"]` y `frag["fuente"]` | Recupera las **palabras reales del documento** y de dónde salieron. |

Si recordás una sola función para el chatbot final, recordá **`buscar_fragmentos()`**.


## Paso 8 — Hacer nuestra primera búsqueda

Probá una pregunta que **sí debería estar respondida en alguno de tus documentos**. Si estás en modo demo, la de inscripción es un ejemplo.

Primero verás **qué fuente recuperó y qué tan relacionada parece**; debajo veremos el texto que encontró.


In [ ]:
pregunta = "¿Qué documentos necesito para inscribirme?"  # Adaptar al tema de tu chatbot

resultado = buscar_fragmentos(pregunta, top_k=3, umbral=0.75)

if resultado.empty:
    print("No se recuperaron fragmentos que superen el umbral elegido.")
    print("No significa necesariamente que la respuesta no exista: revisá la fuente, la pregunta y el umbral.")
else:
    display(resultado[["fuente", "fragmento_numero", "similitud"]])
    for _, fila in resultado.iterrows():
        print("\nFUENTE:", fila["fuente"], "| SIMILITUD:", fila["similitud"])
        print("TEXTO ENCONTRADO:", fila["texto"][:700])

## Paso 9 — ¿Qué ocurre si cambiamos la pregunta?

Una persona puede escribir **“¿Cómo hago la inscripción?”** o **“¿Qué papeles necesito para anotarme?”**. Queremos que el buscador encuentre fuentes parecidas.

**Actividad:** reemplazá las preguntas por **cinco preguntas de tu proyecto**: tres que tus documentos sí deberían poder contestar y dos que no deberían poder contestar o sean dudosas.

Un resultado **no se considera correcto solo por tener una similitud alta**: leé el fragmento y revisá si aporta la información necesaria.


In [ ]:
preguntas_prueba = [
    "¿Qué documentos necesito para inscribirme?",
    "¿Cómo hago para anotarme?",
    "¿Dónde consulto los horarios de Secretaría?",
    "¿Cuánto cuesta viajar a Marte?",  # fuera del tema en el ejemplo
    "¿Cuándo termina la inscripción?"   # requiere fuente actualizada
]

resultados_todos = []
for consulta in preguntas_prueba:
    encontrados = buscar_fragmentos(consulta, top_k=3, umbral=0.75)
    if encontrados.empty:
        resultados_todos.append({
            "pregunta": consulta, "fuente": "sin resultados",
            "fragmento_numero": "-", "similitud": None, "texto": ""
        })
    else:
        resultados_todos.extend(encontrados.to_dict("records"))

df_resultados = pd.DataFrame(resultados_todos, columns=COLUMNAS_RESULTADOS)
display(df_resultados[["pregunta", "fuente", "fragmento_numero", "similitud"]])


## Paso 10 — Comparar `top_k`: ¿cuántos fragmentos conviene recuperar?

Probamos la **misma pregunta** cambiando solo `top_k`.

- **Pocos fragmentos:** menos texto, más rapidez, pero puede faltar algo importante.
- **Muchos fragmentos:** más información, pero también ruido y respuestas potencialmente confusas.

**Mirá cuántos resultados trae y si las fuentes están relacionadas**, no solo los puntajes.


In [ ]:
pregunta_k = "¿Qué necesito para anotarme?"

for k in [1, 2, 4, 8]:
    resultados_k = buscar_fragmentos(pregunta_k, top_k=k, umbral=0.75)
    print("\ntop_k =", k, "| Fragmentos aceptados:", len(resultados_k))
    display(resultados_k[["fuente", "fragmento_numero", "similitud"]])

## Paso 11 — Comparar `umbral`: ¿cuánto parecido vamos a exigir?

Probamos una consulta **posiblemente fuera del tema** (adaptala si tu chatbot trata de ese asunto).

- Umbral **más bajo:** pueden aparecer fragmentos vagamente relacionados.
- Umbral **más alto:** pueden desaparecer todos los resultados.

> Aunque recuperemos un fragmento, **NO significa que ese fragmento responda bien** ni que debamos inventar algo para completar la respuesta.

**Con E5 los puntajes suelen ser relativamente altos.** Por eso probamos `0.70` a `0.90` en vez de límites muy bajos. Si todos los resultados pasan, subí el umbral; si no aparece ninguno, bajalo, pero **siempre revisando el texto**.


In [ ]:
pregunta_umbral = "¿Quién ganó el mundial de fútbol?"

for minimo in [0.70, 0.75, 0.80, 0.85, 0.90]:
    resultados_umbral = buscar_fragmentos(pregunta_umbral, top_k=3, umbral=minimo)
    print("\numbral =", minimo, "| Fragmentos aceptados:", len(resultados_umbral))
    display(resultados_umbral[["fuente", "similitud"]])

# PARTE C — CÓMO ESTO SE CONECTARÁ CON EL CHATBOT

## Paso 12 — De fragmentos recuperados a **contexto para la IA**

Ya conseguimos lo más importante: **recuperar texto original de los documentos**.

Ahora vamos a **juntar esos fragmentos** en un bloque llamado `contexto`. **Este contexto es lo que más adelante pasaremos al modelo generativo**, junto con la pregunta.

En esta celda todavía **no llamamos a Qwen** y **no producimos la respuesta final**. Solo preparamos los datos que verá la IA.


In [ ]:
def construir_contexto(resultados: pd.DataFrame) -> str:
    """Arma el contexto documental con fuente y texto recuperado para el futuro RAG."""
    if resultados.empty:
        return ""

    partes = []
    for numero, (_, fila) in enumerate(resultados.iterrows(), start=1):
        partes.append(
            f"[Fragmento {numero} | Fuente: {fila['fuente']}]\n{fila['texto']}"
        )
    return "\n\n".join(partes)


pregunta_usuario = "¿Qué documentos necesito para inscribirme?"
fragmentos_encontrados = buscar_fragmentos(pregunta_usuario, top_k=3, umbral=0.75)
contexto_para_ia = construir_contexto(fragmentos_encontrados)

print("PREGUNTA:", pregunta_usuario)
print("\nCONTEXTO RECUPERADO (esto recibirá la IA):")
print(contexto_para_ia if contexto_para_ia else "Sin fragmentos suficientes.")

## Paso 13 — ¿Cómo sería el mensaje que recibirá el modelo generativo?

En nuestras clases anteriores ya utilizamos un **prompt de sistema**. En el chatbot final vamos a combinar:

1. **Reglas del sistema:** responder en español, no inventar, citar fuentes.
2. **Contexto documental:** fragmentos recuperados por la función que construimos.
3. **Pregunta original:** lo que escribió el usuario.

Así usamos **PLN + fuentes + IA generativa**.

**Esto es una vista previa, NO es una ejecución del modelo generativo:**


In [ ]:
def preparar_mensajes_rag(pregunta: str, contexto: str) -> list[dict]:
    instrucciones = (
        "Sos un asistente institucional. Respondé en español claro y breve. "
        "Usá únicamente la información del CONTEXTO DOCUMENTAL para datos institucionales. "
        "No inventes fechas, requisitos, teléfonos ni reglamentos. "
        "Si los fragmentos no permiten responder, decí que no hay información suficiente. "
        "Cuando uses un fragmento, mencioná el nombre de su fuente."
    )
    return [
        {"role": "system", "content": instrucciones},
        {"role": "user", "content": f"CONTEXTO DOCUMENTAL:\n{contexto}\n\nPREGUNTA:\n{pregunta}"}
    ]

if contexto_para_ia:
    mensajes_para_qwen = preparar_mensajes_rag(pregunta_usuario, contexto_para_ia)
    print("El buscador ya preparó los mensajes que la IA necesitará después:")
    for mensaje in mensajes_para_qwen:
        print("\nROL:", mensaje["role"])
        print(mensaje["content"][:2200])
else:
    mensajes_para_qwen = []
    print("No encontramos evidencia suficiente: NO deberíamos pedirle al modelo que invente una respuesta.")

### Entonces, ¿qué hemos construido realmente?

**Ya funciona en esta clase:**

```text
pregunta del usuario
    ↓
buscar_fragmentos(pregunta)
    ↓
textos relevantes + nombres de fuentes
    ↓
construir_contexto(resultados)
    ↓
preparar_mensajes_rag(pregunta, contexto)
```

**Aún falta para el chatbot completo (clase 11):**

```text
mensajes preparados
    ↓
MODELO GENERATIVO (Qwen u otro)
    ↓
respuesta redactada usando el contexto
    ↓
interfaz del chatbot + fuentes
```

**Qué código deben conservar para la clase 11:** carga de fragmentos, modelo de embeddings, embeddings de fragmentos, `buscar_fragmentos()` y `construir_contexto()`. Las tablas de pruebas y el CSV son para evaluar, **no son la respuesta final**.

En el Colab de la clase 11 podremos volver a calcular los embeddings al iniciar la sesión; si implementáramos una aplicación permanente, lo habitual sería guardarlos en un índice para no recalcularlos innecesariamente.

**Importante al pasar a la clase 11:** trasladá también el `top_k` y el `umbral` que el grupo haya probado aquí. No conviene dejar por costumbre un umbral genérico muy bajo para E5; deben elegirlo después de evaluar sus fuentes.


# PARTE D — EVALUACIÓN DE LA BÚSQUEDA

## Paso 14 — Comprobar si las fuentes recuperadas sirven

**La computadora da una puntuación, pero nosotros tenemos que analizar la calidad.**

Por cada pregunta, revisá:

- **Correcto:** el fragmento realmente ayuda a responder la consulta.
- **Dudoso:** el fragmento toca el tema pero le falta la respuesta o una fuente confiable.
- **Incorrecto:** no está relacionado o tiene información que no corresponde.

**Ejemplo:** si un fragmento dice *“las fechas se publican en el calendario oficial”*, eso **no permite confirmar una fecha concreta** aunque haya dado mucha similitud.

Usá la siguiente tabla y **completá las evaluaciones en el código** antes de exportar.


In [ ]:
df_evaluacion = df_resultados.copy()
df_evaluacion["evaluacion"] = "pendiente"  # Reemplazar por: correcto / dudoso / incorrecto


**Cómo completar tu evaluación:** cada fila tiene un número de índice (0, 1, 2...). Podés editar las celdas de la tabla desde el código, por ejemplo:

```python
df_evaluacion.loc[0, "evaluacion"] = "correcto"
df_evaluacion.loc[0, "observacion"] = "Trae el documento de inscripción y menciona los requisitos"
```

La palabra `correcto` debe basarse en leer el contenido, no solo en el valor de similitud.


In [ ]:
df_evaluacion["observacion"] = ""

# COMPLETÁ TUS OBSERVACIONES aquí. Ejemplo (descomentá y editá):
# df_evaluacion.loc[0, "evaluacion"] = "correcto"
# df_evaluacion.loc[0, "observacion"] = "Este fragmento sí ayuda a responder."
# df_evaluacion.loc[1, "evaluacion"] = "dudoso"
# df_evaluacion.loc[1, "observacion"] = "Habla del tema, pero no da el dato que pide."

display(df_evaluacion[["pregunta", "fuente", "similitud", "evaluacion", "observacion"]])


## Paso 15 — Guardar el informe de evaluación

El CSV que exportamos a continuación es **una tabla para revisar cómo funcionó la búsqueda**.

- **NO es la base de conocimiento de nuestro chatbot.**
- **NO reemplaza a los PDFs originales ni al `fragmentos_proyecto.json`.**
- **NO contiene una respuesta redactada por la IA.**

En la próxima clase volveremos a usar los fragmentos y la búsqueda para completar el sistema RAG.


In [ ]:
salida_csv = "evaluacion_busqueda_semantica_clase10.csv"
df_evaluacion.to_csv(salida_csv, index=False, encoding="utf-8-sig")
print("CSV de evaluación generado:", salida_csv)

# Si estás en Google Colab, descomentá las dos líneas siguientes para descargar:
# from google.colab import files
# files.download(salida_csv)


## Actividad final y entrega

**Trabajo sobre el chatbot de cada grupo:**

1. Cargá el JSON de tus fragmentos de la clase 9 (o, si te faltan fuentes, realizá la práctica con el modo demo y aclaralo).
2. Escribí **5 preguntas**: 3 que deberían estar cubiertas por los documentos y 2 que no.
3. Probá `top_k=2` y `top_k=4`. Observá si aparece información de más o si falta contexto.
4. Probá **dos valores de umbral** y explicá qué cambió.
5. Revisá las fuentes recuperadas: marcá resultados **correctos, dudosos o incorrectos**, justificando con el contenido.
6. **Explicá en 4 o 5 renglones** qué hace el modelo de embeddings y qué función reutilizaremos en el chatbot.

**Entregar:** `evaluacion_busqueda_semantica_clase10.csv` y una breve conclusión con los valores elegidos de `top_k` y `umbral`.

### Preguntas para cerrar la clase

- ¿El modelo de embeddings escribe la respuesta del chatbot? **No.**
- ¿Un puntaje alto significa que la fuente es verdadera? **No.**
- ¿Qué función encuentra los fragmentos? **`buscar_fragmentos()`**.
- ¿Qué le enviaremos al modelo generativo en la clase 11? **La pregunta y el contexto formado por los fragmentos recuperados**.
- ¿Qué pasa si no hay evidencia? **El chatbot no debería inventar datos específicos**.

**Frase final:** *Hoy construimos la parte que encuentra la información. En la siguiente clase construiremos la parte que la usa para responder.*
